# 🧠 EEG Nonlinear & RQA Analysis Pipeline (Google Colab High-RAM + Google Drive)

本 Notebook 专为在 **Google Colab High-RAM (53GB RAM) + GPU 实例** 上运行非线性 EEG 特征提取与递归分析 (RQA) 设计，结合用户的 **Google AI 5TB (Google Drive)** 存储空间实现冷热存储分离与断点续跑。

### 🌟 核心架构与加速策略
1. **冷热存储分离**：
   - **冷存储 (Google Drive 5TB)**：存放原始 `.mat` 压缩包和最终产物（`_features.json` / `_NL_Results.mat`）。
   - **热计算 (Colab 本地 NVMe SSD)**：数据解压到 `/content/local_data`，临时切片缓存放在 `/content/sig_cache`，**彻底避开 Google Drive FUSE 挂载盘导致的 I/O 拥堵、mmap 锁死和 API 限流问题**。
2. **自适应算力调优**：
   - 自动匹配 Colab 53GB 宿主内存与 GPU 显存，将并发 worker 配额从本地保守的 `0.00005` 提升至高效的 `0.5 ~ 0.7`。
3. **CuPy GPU 原生加速**：
   - 自动安装配置 `cupy-cuda12x`，使得 RQA 距离矩阵运算提速 10x ~ 50x。
4. **断点续跑与实时安全回传**：
   - 开启 `--skip-existing`，即使因网络波动断开重连，也绝不重复计算已完成的 Subject；
   - 每完成一个 Subject，立即实时同步结果至 Google Drive。

## 1. 硬件规格与运行时自检
运行前请确保：在顶部菜单栏依次点击 **Runtime (代码执行程序) -> Change runtime type (更改运行时类型)**：
- **Hardware accelerator (硬件加速器)**: 选择 **GPU** (推荐 A100 / L4 / T4) 或 **None (CPU)**
- **Runtime shape (运行时配置)**: 选择 **High-RAM (大内存，~53GB)**

In [ ]:
import os, sys, psutil

print("=" * 60)
print("🖥️  Colab 硬件与内存环境检测")
print("=" * 60)

# 1. CPU & 宿主物理内存检测
cpu_count = os.cpu_count()
vm = psutil.virtual_memory()
total_ram_gb = vm.total / (1024 ** 3)
avail_ram_gb = vm.available / (1024 ** 3)
print(f"CPU 核心数: {cpu_count} vCPUs")
print(f"系统总内存: {total_ram_gb:.2f} GB (可用: {avail_ram_gb:.2f} GB)")
if total_ram_gb < 30:
    print("⚠️ 警告: 当前未启用 High-RAM 实例 (内存 < 30GB)。建议在 Runtime 设置中切换为 High-RAM 获得 53GB 内存！")
else:
    print("✅ High-RAM 实例已就绪 (内存 ~53GB)！")

# 2. GPU 检测
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "当前未挂载 GPU (纯 CPU 模式)"

## 2. 挂载 Google Drive (连接 5TB 云存储)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 定义 Google Drive 上的持久化工作目录
DRIVE_WORKSPACE = "/content/drive/MyDrive/EEG_Pipeline"
DRIVE_DATA_ARCHIVE = f"{DRIVE_WORKSPACE}/eeg_data_colab.tar.gz"
DRIVE_OUTPUT_DIR = f"{DRIVE_WORKSPACE}/output_data"

os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"✅ Google Drive 工作目录就绪: {DRIVE_WORKSPACE}")

## 3. 克隆代码仓库并安装 Linux/CUDA 依赖
自动配置 CuPy (CUDA 12) 以及本项目所需的全部非线性与信号处理库。

In [ ]:
REPO_DIR = "/content/eeg-nonlinear-pipeline"

# 1. 获取代码 (若未克隆则从 GitHub 拉取)
if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    !git clone https://github.com/TH-yi/eeg-nonlinear-pipeline.git {REPO_DIR}
else:
    print("Repository already exists, pulling latest updates...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# 2. 安装基础依赖
!pip install -q typer mat73 nolds antropy mne psutil scipy rich

# 3. 安装 CuPy (根据 CUDA 驱动自动适配 GPU 距离矩阵运算)
!pip install -q cupy-cuda12x

print("✅ 依赖库安装配置完成！")

## 4. 解压数据至 Colab 本地 NVMe 高速 SSD
将从本地上传到 Google Drive 的 `eeg_data_colab.tar.gz` 瞬间解压到本地热计算盘 `/content/local_data`。

In [ ]:
LOCAL_DATA_DIR = "/content/local_data"
LOCAL_OUTPUT_DIR = "/content/local_output"
LOCAL_CACHE_DIR = "/content/sig_cache"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

# 检查本地数据是否已存在，不存在则从 Drive 解压
local_files = [f for f in os.listdir(LOCAL_DATA_DIR) if f.endswith('.mat')]
if len(local_files) == 0:
    if os.path.exists(DRIVE_DATA_ARCHIVE):
        print(f"📦 正在从 Google Drive 解压数据包: {DRIVE_DATA_ARCHIVE} -> {LOCAL_DATA_DIR} ...")
        !tar -xzf {DRIVE_DATA_ARCHIVE} -C {LOCAL_DATA_DIR}
        mats = [f for f in os.listdir(LOCAL_DATA_DIR) if f.endswith('.mat')]
        print(f"✅ 解压完成！共发现 {len(mats)} 个 Subject .mat 文件在本地高速盘")
    else:
        print(f"⚠️ 未在 Google Drive 找到压缩包: {DRIVE_DATA_ARCHIVE}")
        print("请先在本地运行 `python scripts/pack_colab_data.py`，并将生成的 `eeg_data_colab.tar.gz` 拖入 Google Drive 的 EEG_Pipeline 文件夹！")
else:
    print(f"✅ 本地高速盘已有 {len(local_files)} 个 Subject 数据，直接复用无需再次解压！")

# 同步 Google Drive 已有的输出产物到本地 (以便断点续跑跳过)
if os.path.exists(DRIVE_OUTPUT_DIR):
    !cp -n {DRIVE_OUTPUT_DIR}/*_features.json {LOCAL_OUTPUT_DIR}/ 2>/dev/null || true
    existing_done = len([f for f in os.listdir(LOCAL_OUTPUT_DIR) if f.endswith('_features.json')])
    print(f"ℹ️ 已从 Google Drive 同步已完成的 {existing_done} 个被试结果用于断点跳过。")

## 5. 配置 High-RAM 资源并启动流水线
可灵活选择计算模式：
- `METHOD = 'rqa'`: 递归分析（推荐开启 GPU，计算耗时仅几十秒/被试）
- `METHOD = 'nonlinear'`: 非线性动力学特征（CPU 高并发模式）

In [ ]:
import os, subprocess, time, glob, shutil

# ================== 运行参数配置 ==================
METHOD = "rqa"          # 'rqa' 或 'nonlinear'
USE_GPU = True          # RQA 模式推荐开启 GPU
SKIP_EXISTING = True    # 开启断点续跑（跳过已处理被试）
# ==================================================

# 环境变量注入 (释放 53GB High-RAM 算力，告别本地 0.00005 极端保守限制)
os.environ["EEG_MEM_LIMIT"] = "0.5"           # 允许使用 50% 可用内存作为安全预算
os.environ["EEG_CPU_RATIO"] = "0.7"           # 允许调动 70% CPU 核心
os.environ["EEG_PARALLEL_TASKS"] = "4"        # 允许通道级 4 线程并发
os.environ["EEG_CACHE_DIR"] = LOCAL_CACHE_DIR  # 临时切片缓存严格限定在本地高速盘

print(f"🚀 启动 EEG 分析流水线: Method={METHOD} | GPU={USE_GPU} | SkipExisting={SKIP_EXISTING}")

# 构造启动指令
cmd = [
    sys.executable, "main.py", "process",
    "--data-dir", LOCAL_DATA_DIR,
    "--output-dir", LOCAL_OUTPUT_DIR,
    "--method", METHOD,
    "--use-gpu" if USE_GPU else "--no-use-gpu",
    "--skip-existing" if SKIP_EXISTING else "--force"
]

start_time = time.time()

# 启动进程并实时监听完成事件，自动同步回 Google Drive 5TB
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

synced_files = set(os.listdir(DRIVE_OUTPUT_DIR))

try:
    for line in proc.stdout:
        print(line, end="")
        # 检查是否有新产出的 features.json 并立即同步至 Drive (实现增量安全备份)
        current_files = set(glob.glob(f"{LOCAL_OUTPUT_DIR}/*_features.json") + glob.glob(f"{LOCAL_OUTPUT_DIR}/*_NL_Results.mat"))
        for fpath in current_files:
            fname = os.path.basename(fpath)
            if fname not in synced_files:
                shutil.copy2(fpath, os.path.join(DRIVE_OUTPUT_DIR, fname))
                synced_files.add(fname)
                print(f"[☁️ 云端备份] 成功同步 {fname} 至 Google Drive!")
except KeyboardInterrupt:
    print("⚠️ 收到中断信号，正在安全停止...")
    proc.terminate()

proc.wait()
elapsed = time.time() - start_time

# 最终完整同步所有汇总文件 (Creativity_NL_Data.json / .mat)
!cp -r {LOCAL_OUTPUT_DIR}/* {DRIVE_OUTPUT_DIR}/

print("=" * 60)
print(f"🎉 全部处理与同步完成！总耗时: {elapsed / 60:.2f} 分钟")
print(f"📁 结果已持久化保存至 Google Drive: {DRIVE_OUTPUT_DIR}")
print("=" * 60)

## 6. 结果快速检验与统计预览

In [ ]:
import json, glob
import scipy.io as sio

results = sorted(glob.glob(f"{DRIVE_OUTPUT_DIR}/*_features.json"))
print(f"📊 Google Drive 中已完成特征提取的 Subject 数量: {len(results)}")
for r in results[:5]:
    print(f"  - {os.path.basename(r)}")
if len(results) > 5:
    print(f"  ... 以及另外 {len(results) - 5} 个被试")

# 检查全局汇聚 MAT 文件
agg_mat = f"{DRIVE_OUTPUT_DIR}/Creativity_NL_Data.mat"
if os.path.exists(agg_mat):
    data = sio.loadmat(agg_mat)
    print("\n✅ 全局汇聚 MAT 文件正常:")
    for k in data:
        if not k.startswith("__"):
            print(f"  Key: {k}, Shape: {data[k].shape}")